In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import random
import copy
from torch.utils.data import Dataset, DataLoader

In [2]:
# --------- Sudoku Utilities ----------
def is_valid(grid, row, col, num): # Function to validate if a number can be placed into a specific cell without violating Sudoku rules
    for i in range(9):
        if grid[row][i] == num or grid[i][col] == num:
            return False # Reurns false if number already exxists in row/column
    r, c = 3 * (row // 3), 3 * (col // 3)
    for i in range(3):
        for j in range(3):
            if grid[r + i][c + j] == num:
                return False # Returns false if number already exists in 3 x 3 subgrid
    return True # Returns true if valid number

def solve(grid): # Backtracking Algorithm to solve Sudoku recursively
    for i in range(9):
        for j in range(9):
            if grid[i][j] == 0:
                for num in range(1, 10):
                    if is_valid(grid, i, j, num):
                        grid[i][j] = num
                        if solve(grid): return True
                        grid[i][j] = 0
                return False
    return True

def generate_sudoku(): # Generates fully solved Sudoku grids
    grid = [[0] * 9 for _ in range(9)]
    def is_valid(r, c, num):
        for i in range(9):
            if grid[r][i] == num or grid[i][c] == num:
                return False
        sr, sc = 3 * (r // 3), 3 * (c // 3)
        for i in range(sr, sr + 3):
            for j in range(sc, sc + 3):
                if grid[i][j] == num:
                    return False
        return True
        
    def fill(): # Randomise number selection for each puzzle
        for r in range(9):
            for c in range(9):
                if grid[r][c] == 0:
                    nums = list(range(1, 10))
                    random.shuffle(nums)
                    for num in nums:
                        if is_valid(r, c, num):
                            grid[r][c] = num
                            if fill():
                                return True
                            grid[r][c] = 0
                    return False
        return True
    fill()
    return grid

In [3]:
# --------- Dataset ----------
class SudokuDataset(Dataset): # Generate training data for Model
    def __init__(self, size=100, clue_range=(20, 40)): # Default size and range of clues for variety of training data
        self.samples = []
        for _ in range(size): # Use generated Sudoku and create puzzle by removing specified number of digits
            full = generate_sudoku()
            puzzle = copy.deepcopy(full)
            indices = [(i, j) for i in range(9) for j in range(9)]
            random.shuffle(indices)
            clues = random.randint(*clue_range)
            for i, j in indices[:81 - clues]: # Remove number of 'Clues' for varying difficulty
                puzzle[i][j] = 0
            self.samples.append((np.array(puzzle), np.array(full)))
    
    def __len__(self): # Return number of samples in dataset
        return len(self.samples) 
    
    def __getitem__(self, idx): # Function to process and return data when model requests a batch
        puzzle, solution = self.samples[idx]
        
        # One-hot encode the puzzle input
        x = torch.zeros((10, 9, 9), dtype=torch.float32)  # Channel 0 for empty cells, 1-9 for digits
        for i in range(9):
            for j in range(9):
                digit = puzzle[i][j]
                x[digit, i, j] = 1.0  # One-hot encoding
        
        y = torch.tensor(solution - 1, dtype=torch.long)  # Correct solution, 9 x 9 shape
        mask = torch.tensor(puzzle != 0)  # Boolean mask of clues positions within puzzle, 9 x 9 shape
        return x, y, mask

In [4]:
# --------- Model ----------
class SudokuCNN(nn.Module):
    def __init__(self): 
        super().__init__()
        # Input: 10 channels (0 + digits 1-9 one-hot encoded)
        self.conv1 = nn.Conv2d(10, 64, kernel_size=3, padding=1) # Applies 64 filters to input, padding to preserve 9 x 9 size
        self.bn1 = nn.BatchNorm2d(64) # Normalise outputs for faster training
        
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # Define 5 Residual blocks 
        self.residual_blocks = nn.ModuleList([
            ResidualBlock(64) for _ in range(5)
        ])
        
        self.conv_out = nn.Conv2d(64, 9, kernel_size=1) # Output layer
    
    def forward(self, x):
        # Initial convolutions using ReLU
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        
        # Apply residual blocks in sequence
        for block in self.residual_blocks:
            x = block(x)
        
        # Convert 64 features into class scores for each cell
        x = self.conv_out(x)  # (B, 9, 9, 9)
        
        # Reshape to the desired output format
        return x.permute(0, 2, 3, 1)  # (B, 9, 9, 9)

In [5]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1) # Convolution layers, padding to preserve size
        self.bn1 = nn.BatchNorm2d(channels) # Normalisation after each convolution
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
    
    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x))) # Convolutions using ReLU
        x = self.bn2(self.conv2(x))
        x += residual  # Skip connection to improve training
        x = F.relu(x)
        return x

def sudoku_constraints_loss(pred_logits, target): # Calulate penalties for violating constraints
    """
    Apply Sudoku constraints to the predicted logits.
    pred_logits: (batch_size, 9, 9, 9) - logits for each digit for each cell
    target: (batch_size, 9, 9) - target values (0-8)
    """
    batch_size = pred_logits.size(0)
    
    # Convert logit probabilities for each digit per cell
    pred_probs = F.softmax(pred_logits, dim=3)  # (B, 9, 9, 9)
    
    loss = 0.0
    
    # Row constraint: each digit should appear once in each row
    row_probs = pred_probs.sum(dim=1)  # (B, 9, 9)
    row_target = torch.ones(batch_size, 9, 9, device=pred_logits.device)
    row_loss = F.mse_loss(row_probs, row_target) # Penalty if false
    
    # Column constraint: each digit should appear once in each column
    col_probs = pred_probs.sum(dim=2)  # (B, 9, 9)
    col_target = torch.ones(batch_size, 9, 9, device=pred_logits.device)
    col_loss = F.mse_loss(col_probs, col_target) # Penalty if false
    
    # Box constraint: each digit should appear once in each 3x3 box
    box_loss = 0.0
    for i in range(0, 9, 3):
        for j in range(0, 9, 3):
            box_probs = pred_probs[:, i:i+3, j:j+3, :].reshape(batch_size, 9, 9)
            box_target = torch.ones(batch_size, 9, 9, device=pred_logits.device)
            box_loss += F.mse_loss(box_probs, box_target) # Penalty if false
    box_loss /= 9.0  # Average over the 9 boxes
    
    # Combine for total penalty value
    constraint_loss = row_loss + col_loss + box_loss 
    return constraint_loss

In [6]:
# --------- Training ----------
def train_model(epochs=15, lr=0.001, weight_decay=1e-5, batch_size=32, dataset_size=10000): # Initialise model 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SudokuCNN().to(device)
    print("\n========== TRAINING STARTED ==========")
    print(f"Using device: {device}")
    print(f"Training with {dataset_size} examples, batch size {batch_size}")
    print(f"Initial learning rate: {lr}")
    print("=======================================\n")
    
    # Generate training dataset
    print("Generating training dataset...")
    train_dataset = SudokuDataset(size=dataset_size, clue_range=(30, 40))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)  # Reduced workers
    
    # Generate seperate validation dataset
    print("Generating validation dataset...")
    val_dataset = SudokuDataset(size=500, clue_range=(30, 40)) 
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)  # Reduced workers
    
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay) # Adam optimiser fro adaptive learning, weight decay to avoid overfitting
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)
    ce_loss_fn = nn.CrossEntropyLoss(reduction='none') # Cross entropy for digit prediction at each cell

    # Intial values
    best_val_acc = 0.0
    best_model = None
    
    for epoch in range(1, epochs + 1): # Loop for specified num of epochs
        model.train()
        total_loss = 0.0
        train_correct = 0
        train_total = 0
        batches = 0
        
        print(f"Starting epoch {epoch}/{epochs}")
        
        for x, y, mask in train_loader:
            x, y, mask = x.to(device), y.to(device), mask.to(device)
            
            optimizer.zero_grad()
            out = model(x)  # (B, 9, 9, 9)
            
            # Calculate standard cross-entropy loss
            out_for_ce = out.permute(0, 3, 1, 2)  # -> (B, 9_digits, 9, 9)
            loss_tensor = ce_loss_fn(out_for_ce, y)  # (B, 9, 9)
            
            # Mask for unknown cells
            unknown_mask = ~mask  # True where cell is unknown
            masked_loss = loss_tensor * unknown_mask
            ce_loss = masked_loss.sum() / unknown_mask.sum().clamp(min=1) # Clamp to prevent division by 0
            
            # Add Sudoku constraints loss
            constraint_loss = sudoku_constraints_loss(out, y)
            
            # Total loss with weighted constraints
            loss = ce_loss + 0.4 * constraint_loss
            
            # Compute gradients and backpropagation
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Gradient clipping to prevent instability
            optimizer.step()
            
            # Detailed loss tracking
            ce_loss_value = ce_loss.item()
            constraint_loss_value = constraint_loss.item()
            
            total_loss += loss.item()
            batches += 1
            
            # Print batch progress periodically
            if batches % 100 == 0: 
                print(f"  Batch {batches}: CE Loss: {ce_loss_value:.6f}, Constraint Loss: {constraint_loss_value:.6f}, Total: {loss.item():.6f}")
            
            # Calculate accuracy on unknown cells
            with torch.no_grad():
                pred = torch.argmax(out, dim=3)  # (B, 9, 9)
                correct = ((pred == y) & unknown_mask).sum().item()
                total = unknown_mask.sum().item()
                train_correct += correct
                train_total += total
        
        train_loss = total_loss / batches # Training loss averaged over batches
        train_acc = train_correct / train_total if train_total > 0 else 0.0 # Training accuracy over total
        
        # Validation phase
        print("Starting validation...")
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        val_batches = 0
        
        with torch.no_grad(): # Use no_grad() for improved memory usage and speed
            for x, y, mask in val_loader:
                x, y, mask = x.to(device), y.to(device), mask.to(device)
                
                out = model(x)
                
                # Same loss calculation as training
                out_for_ce = out.permute(0, 3, 1, 2)
                loss_tensor = ce_loss_fn(out_for_ce, y)
                unknown_mask = ~mask
                masked_loss = loss_tensor * unknown_mask
                ce_loss = masked_loss.sum() / unknown_mask.sum().clamp(min=1)
                
                constraint_loss = sudoku_constraints_loss(out, y)
                loss = ce_loss + 0.4 * constraint_loss
                
                val_loss += loss.item()
                val_batches += 1
                
                # Calculate accuracy
                pred = torch.argmax(out, dim=3)  # (B, 9, 9)
                correct = ((pred == y) & unknown_mask).sum().item()
                total = unknown_mask.sum().item()
                val_correct += correct
                val_total += total
        
        val_loss = val_loss / val_batches if val_batches > 0 else 0 # Validation loss averaged over batches
        val_acc = val_correct / val_total if val_total > 0 else 0.0 # Validation accuracy over total
        
        # Update learning rate based on validation loss
        scheduler.step(val_loss)
        
        # Detailed print statements for monitoring training progress
        print(f"Epoch {epoch:2d}/{epochs}")
        print(f"  Training:   Loss: {train_loss:.6f} | Accuracy: {train_acc:.6f}")
        print(f"  Validation: Loss: {val_loss:.6f} | Accuracy: {val_acc:.6f}")
        print(f"  Learning rate: {optimizer.param_groups[0]['lr']:.8f}")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = copy.deepcopy(model.state_dict())
            print(f"  New best model with validation accuracy: {best_val_acc:.6f}")
        
        print("---------------------------------------")
    
    # Load best model
    if best_model is not None:
        model.load_state_dict(best_model)
    print("\n========== TRAINING COMPLETED ==========")
    print(f"Best validation accuracy: {best_val_acc:.6f}")
    print("========================================\n")
    return model # Return best model for final evaluation

In [7]:
# --------- Inference ----------
def solve_with_model(model, puzzle): # Solve puzzle with trained model and return solution
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    
    # Create a copy of the puzzle to modify, original puzzle unchanged
    solution = np.copy(puzzle)
    puzzle_np = np.array(puzzle)
    
    # One-hot encode the puzzle input
    x = torch.zeros((1, 10, 9, 9), dtype=torch.float32)
    for i in range(9):
        for j in range(9):
            digit = puzzle[i][j]
            x[0, digit, i, j] = 1.0
    
    x = x.to(device)
    
    # Boolean mask for clues
    fixed_mask = torch.tensor(puzzle_np != 0).to(device)
    
    with torch.no_grad():
        # Get model predictions
        logits = model(x)  # (1, 9, 9, 9)
        
        # For cells that aren't clues, select most likely digit
        pred = torch.argmax(logits[0], dim=2).cpu().numpy() + 1  # Add 1 because digit predictions are 0-8
        
        # Update only non-clue cells
        for i in range(9):
            for j in range(9):
                if puzzle[i][j] == 0: 
                    solution[i][j] = pred[i][j]
    
    return solution

def evaluate_model(model, num_puzzles=100, clue_range=(30, 40)): # Test model on num of puzzles and calculate accuracy
    print("\n========== MODEL EVALUATION ==========")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    # Initial values
    correct_cells = 0
    total_cells = 0
    complete_correct = 0
    
    print(f"Evaluating on {num_puzzles} puzzles...")
    
    # Create puzzles to specifications
    for i in range(num_puzzles): 
        # Generate a puzzle
        solution = generate_sudoku()
        puzzle = copy.deepcopy(solution)
        indices = [(i, j) for i in range(9) for j in range(9)]
        random.shuffle(indices)
        clues = random.randint(*clue_range)
        for i, j in indices[:81 - clues]:
            puzzle[i][j] = 0
        
        # Solve with model
        prediction = solve_with_model(model, puzzle)
        
        # Calculate cell accuracy
        unknown_mask = (np.array(puzzle) == 0)
        correct = np.sum((np.array(prediction) == np.array(solution)) & unknown_mask)
        total = np.sum(unknown_mask)
        
        correct_cells += correct
        total_cells += total
        
        # Complete puzzle accuracy
        if np.all(np.array(prediction) == np.array(solution)):
            complete_correct += 1
            
        # Print individual puzzle results
        print(f"Puzzle {i+1}: {correct}/{total} correct cells ({(correct/total)*100:.2f}%)")
    
    cell_accuracy = correct_cells / total_cells if total_cells > 0 else 0 # Total cell accuracy
    puzzle_accuracy = complete_correct / num_puzzles # Total number of entire puzzles solved
    
    print(f"Cell Accuracy: {cell_accuracy:.6f} ({correct_cells}/{total_cells} cells correct)")
    print(f"Complete Puzzle Accuracy: {puzzle_accuracy:.6f} ({complete_correct}/{num_puzzles} puzzles correct)")
    print("======================================\n")
    
    return cell_accuracy, puzzle_accuracy

def print_grid(grid, title=""): # Helper function to display Sudoku grids
    print(title)
    for row in grid:
        print(" ".join(str(int(v)) if v != 0 else "." for v in row))
    print()

In [8]:
def main():
    print("Starting the Sudoku CNN solver...")
    
    # Train model with given parameters
    model = train_model(epochs=30, dataset_size=100000, batch_size=64)
    torch.save(model.state_dict(), "sudoku_cnn_model.pth") # Save trained weights
    
    # Evaluate on test puzzles
    evaluate_model(model, num_puzzles=5)
    
    # Example of solving a single puzzle
    print("Generating an example puzzle...")
    solution = generate_sudoku()
    puzzle = copy.deepcopy(solution)
    indices = [(i, j) for i in range(9) for j in range(9)]
    random.shuffle(indices)
    num_cells_to_remove = 40  # Removing fewer cells for better accuracy in this demo
    for i, j in indices[:num_cells_to_remove]:
        puzzle[i][j] = 0
    
    print(f"Created puzzle with {81-num_cells_to_remove} clues and {num_cells_to_remove} empty cells")
    
    print_grid(puzzle, "Puzzle:")
    print("Solving with model...")
    prediction = solve_with_model(model, puzzle)
    print_grid(prediction, "Predicted:")
    print_grid(solution, "Solution:")
    
    # Calculate accuracy for this single puzzle
    unknown_mask = (np.array(puzzle) == 0)
    correct = np.sum((np.array(prediction) == np.array(solution)) & unknown_mask)
    total = np.sum(unknown_mask)
    accuracy = correct / total if total > 0 else 0
    print(f"Cell Accuracy: {accuracy:.2%} ({correct}/{total} cells correct)")
    
    # Detailed error analysis
    print("\nDetailed error analysis:")
    correct_cells = 0
    incorrect_cells = []
    
    for i in range(9):
        for j in range(9):
            if puzzle[i][j] == 0:  # Only check cells that were originally empty
                if prediction[i][j] == solution[i][j]:
                    correct_cells += 1
                else:
                    incorrect_cells.append((i, j, prediction[i][j], solution[i][j]))
    
    print(f"Correct predictions: {correct_cells}/{total}")
    
    if incorrect_cells:
        print(f"Incorrect predictions (row, col, predicted, actual):")
        for r, c, pred, actual in incorrect_cells[:10]:  # Show up to 10 incorrect cells
            print(f"  Position ({r},{c}): Predicted {pred}, Actual {actual}")
        if len(incorrect_cells) > 10:
            print(f"  ... and {len(incorrect_cells) - 10} more incorrect cells")
    
   

In [9]:
if __name__ == "__main__": # Main function
    main()

Starting the Sudoku CNN solver...

========== TRAINING STARTED ==========
Using device: cpu
Training with 100000 examples, batch size 64
Initial learning rate: 0.001

Generating training dataset...
Generating validation dataset...
Starting epoch 1/30
  Batch 50: CE Loss: 2.047606, Constraint Loss: 0.879526, Total: 2.399416
  Batch 100: CE Loss: 1.920180, Constraint Loss: 0.905360, Total: 2.282324
  Batch 150: CE Loss: 1.823211, Constraint Loss: 0.919983, Total: 2.191205
  Batch 200: CE Loss: 1.717094, Constraint Loss: 0.926315, Total: 2.087620
  Batch 250: CE Loss: 1.640436, Constraint Loss: 0.935148, Total: 2.014496
  Batch 300: CE Loss: 1.573161, Constraint Loss: 0.939920, Total: 1.949129
  Batch 350: CE Loss: 1.492155, Constraint Loss: 0.941449, Total: 1.868734
  Batch 400: CE Loss: 1.438683, Constraint Loss: 0.943375, Total: 1.816033
  Batch 450: CE Loss: 1.411444, Constraint Loss: 0.942076, Total: 1.788275
  Batch 500: CE Loss: 1.347847, Constraint Loss: 0.944954, Total: 1.725829
